<a href="https://colab.research.google.com/github/prasadboi/llm-from-scratch/blob/main/DataLoader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Creating input-target pairs

In [4]:
with open("the-verdict.txt", "r", encoding="utf8") as f:
    text = f.read()

# encoding the text using BPE (gpt-2) style tokenizer
import tiktoken
enc = tiktoken.get_encoding("gpt2")
tokens = enc.encode(text)
print(len(tokens))
# for demonstration, delete the first 50 tokens to encounter more interesting cases in the text
tokens = tokens[50:]

5145


In [ ]:
# context size determines how manyy tokens are included in the input
# i.e. how many words the model should pay attention to at a single point in time.

In [7]:
context_size = 4
x = tokens[:context_size]
y = tokens[1:context_size+1]
print(f"input: {x}")
print(f"target:     {y}")

input: [290, 4920, 2241, 287]
target:     [4920, 2241, 287, 257]


In [13]:
# processing the inputs along with the targets, which are the inputs shifted by 1 position, we can then create the next word prediction task as follows
for i in range(1, context_size+1):
    context = tokens[:i]
    desired = tokens[i]
    print(f"input: {context} ---> target: {desired}")
tokenizer = tiktoken.get_encoding("gpt2")
for i in range(1, context_size+1):
    context = tokens[:i]
    desired = tokens[i]
    print(f"input: {tokenizer.decode(context)} ---> target: {tokenizer.decode([desired])}")

input: [290] ---> target: 4920
input: [290, 4920] ---> target: 2241
input: [290, 4920, 2241] ---> target: 287
input: [290, 4920, 2241, 287] ---> target: 257
input:  and ---> target:  established
input:  and established ---> target:  himself
input:  and established himself ---> target:  in
input:  and established himself in ---> target:  a


In [19]:
# Therefore with this logic, to create the input embeddings, we need to implement an efficient dataloader
# given inputs, output target as pytorch tensors (multidim array)
# in particular we are interested in returning 2 tensors: an input tensor contatining the text that the LLM sees and a an output tensor which is what the LLM should predict

# therefore the steps required are as follows:
## tokenize the entire text
## use a sliding window to chunk the book into voverlapping sequences of max_length
## return the total number of rows in the dataset
## return a single row from the dataset

from torch.utils.data import Dataset, DataLoader
class GPTDataset(Dataset):
    def __init__(self, text_file, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text_file, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(input_chunk)
            self.target_ids.append(target_chunk)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# Creating the Dataloader
# steps:
## initialize the toknizer
## create the dataset
## drop_last = True -> drops the alst batch if it is shorter than the specified batch_size to prevent loss spikes during training
## the number of CPU processes to use for preprocessing

def create_dataloader(
        text_file,
        batch_size = 4,
        max_length = 256,
        stride = 128,
        shuffle = True,
        num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(text_file, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers
    )
    # this dataloader method accesses the __getitem__ function of the dataset
    return dataloader

In [21]:
# testing the dataloader with batch size 1
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
# converting the dataloader into a python iterator such that the next entry can be automatically fetched using Python's next() function
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

PyTorch version: 2.10.0+cpu
[[tensor([40]), tensor([367]), tensor([2885]), tensor([1464])], [tensor([367]), tensor([2885]), tensor([1464]), tensor([1807])]]
[[tensor([367]), tensor([2885]), tensor([1464]), tensor([1807])], [tensor([2885]), tensor([1464]), tensor([1807]), tensor([3619])]]


In [23]:
# testing the dataloader with batch size 6
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader(
    raw_text, batch_size=6, max_length=4, stride=1, shuffle=False
)
# converting the dataloader into a python iterator such that the next entry can be automatically fetched using Python's next() function
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"INPUTS:\n\t{inputs}")
print(f"TARGETS:\n\t{targets}")

PyTorch version: 2.10.0+cpu
INPUTS:
	[tensor([  40,  367, 2885, 1464, 1807, 3619]), tensor([ 367, 2885, 1464, 1807, 3619,  402]), tensor([2885, 1464, 1807, 3619,  402,  271]), tensor([ 1464,  1807,  3619,   402,   271, 10899])]
TARGETS:
	[tensor([ 367, 2885, 1464, 1807, 3619,  402]), tensor([2885, 1464, 1807, 3619,  402,  271]), tensor([ 1464,  1807,  3619,   402,   271, 10899]), tensor([ 1807,  3619,   402,   271, 10899,  2138])]
